# COPC Conversion

Convert LAS/LAZ point cloud files to **COPC** (Cloud-Optimized Point Cloud) format.

## What is COPC?

COPC is a standard LAZ 1.4 file with an embedded spatial octree index. This enables:

- **Efficient streaming**: HTTP range requests load only the spatial region needed
- **Fast partial reads**: skip loading the entire file for local visualization
- **Web compatibility**: serve directly to Potree, copc.io viewer, and QGIS 3.26+

COPC files use the `.copc.laz` extension and are fully compatible with all LAS/LAZ tools.

**Requires**: PDAL (included in Docker image and conda environment)

In [ ]:
# Verify PDAL is available
import subprocess
result = subprocess.run(['pdal', '--version'], capture_output=True, text=True)
print(result.stdout or result.stderr)

## Convert a Single File

In [ ]:
from sat.io.las_io import laz_to_copc
from pathlib import Path

# Convert a single LAZ file to COPC
input_file = "/data/output/final_results/example.laz"  # Change to your file
output_file = input_file.replace('.laz', '.copc.laz')

if Path(input_file).exists():
    laz_to_copc(input_file, output_file, verbose=True)
    print(f"\nOriginal: {Path(input_file).stat().st_size / 1e6:.1f} MB")
    print(f"COPC:     {Path(output_file).stat().st_size / 1e6:.1f} MB")
else:
    print(f"File not found: {input_file}")
    print("Run inference first, or change the path to an existing LAZ file.")

## Batch Convert All LAZ Files

In [ ]:
from sat.io.las_io import laz_to_copc
from pathlib import Path

input_dir = Path("/data/output/final_results")  # Change to your directory

laz_files = [f for f in input_dir.glob("*.laz") if '.copc.laz' not in f.name]
print(f"Found {len(laz_files)} LAZ files to convert")

for laz_file in laz_files:
    copc_file = str(laz_file).replace('.laz', '.copc.laz')
    if Path(copc_file).exists():
        print(f"  Skip (exists): {laz_file.name}")
        continue
    try:
        laz_to_copc(str(laz_file), copc_file, verbose=True)
    except Exception as e:
        print(f"  Failed: {laz_file.name} — {e}")

print(f"\nDone. COPC files:")
for f in input_dir.glob("*.copc.laz"):
    print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

## Verify COPC with PDAL

In [ ]:
import subprocess, json
from pathlib import Path

copc_dir = Path("/data/output/final_results")
copc_files = list(copc_dir.glob("*.copc.laz"))

for f in copc_files[:3]:  # Check first 3
    result = subprocess.run(
        ['pdal', 'info', '--summary', str(f)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        info = json.loads(result.stdout)
        summary = info.get('summary', {})
        bounds = summary.get('bounds', {})
        print(f"{f.name}:")
        print(f"  Points: {summary.get('num_points', 'N/A'):,}")
        print(f"  Dimensions: {summary.get('num_dims', 'N/A')}")
        print(f"  Bounds X: [{bounds.get('minx', 'N/A'):.1f}, {bounds.get('maxx', 'N/A'):.1f}]")
        print(f"  Bounds Y: [{bounds.get('miny', 'N/A'):.1f}, {bounds.get('maxy', 'N/A'):.1f}]")
        print()
    else:
        print(f"{f.name}: PDAL error — {result.stderr}")

## Serving COPC Files for Web Viewing

COPC files can be served over HTTP and visualized in a browser:

1. **Start a local HTTP server** in the output directory:
   ```bash
   cd /data/output/final_results
   python -m http.server 8080
   ```

2. **Open the [copc.io viewer](https://viewer.copc.io/)** and enter your file URL:
   ```
   http://localhost:8080/your_file.copc.laz
   ```

3. For **public sharing**, upload your COPC file to any HTTP-accessible storage (S3, GCS, or CyVerse Data Store) and share the direct URL.

The spatial octree index in COPC files means viewers only download the tiles needed for the current view — no need to transfer the entire file.